In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [2]:
import sys
sys.path.append("../src/")

In [3]:
import peaker as pk
from pprint import pprint

In [4]:
in_bed_folder = "/data5/deepro/starrseq/lib_03162022/results/report/cov/IN"
cc_bed_folder = "/data5/deepro/starrseq/lib_03162022/results/report/cov/CC"
ko_bed_folder = "/data5/deepro/starrseq/lib_03162022/results/report/cov/ATF2"

in_beds = [f.path for f in os.scandir(in_bed_folder) if f.path.endswith("filtered_cov.bed")]
cc_beds = [f.path for f in os.scandir(cc_bed_folder) if f.path.endswith("filtered_cov.bed")]
ko_beds = [f.path for f in os.scandir(ko_bed_folder) if f.path.endswith("filtered_cov.bed")]

In [5]:
homer_pwm_motifs = "/data5/deepro/starrseq/lib_03162022/results/analysis/activity_prediction/data/homer/homer.motifs"
roi_bed = "/data5/deepro/starrseq/computational_pipeline/data/roi/master.sorted.bed"
genome_fasta = "/data5/deepro/genomes/hg38/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta"
homer_outdir = "/data5/deepro/starrseq/lib_03162022/results/analysis/activity_prediction/data/homer_out"
homer_pickle = "/data5/deepro/starrseq/lib_03162022/results/analysis/activity_prediction/data/homer_out/motif_features.pkl"

In [6]:
homer_arg_dict = {
    "roi_bed": roi_bed, 
    "genome_fasta": genome_fasta,
    "homer_pwm_motifs": homer_pwm_motifs,
    "homer_outdir": homer_outdir
}

homer_arg_dict = {
    "homer_pickle": homer_pickle
}

In [7]:
# control line peak analysis
cc_peaker = pk.Biopeaker(in_beds, cc_beds)
cc_peaker.homer_featurize(**homer_arg_dict)
cc_model_info = cc_peaker.peaker(labels_of_interest=["low", "high"], high_thresh=1, low_thresh=-1.5)

Loaded pickle ... 
(0.7031007751937984, 0.7045600475195333, 0.6707988980716253, 0.7717908082408875)


In [8]:
# ko line peak analysis
ko_peaker = pk.Biopeaker(in_beds, ko_beds)
ko_peaker.homer_featurize(**homer_arg_dict)
ko_model_info = ko_peaker.peaker(labels_of_interest=["low", "high"], high_thresh=1, low_thresh=-1.5)

Loaded pickle ... 
(0.821917808219178, 0.8243091182428328, 0.7709772951628825, 0.8434125269978402)


In [9]:
cc_a, cc_r, _ = cc_model_info
ko_a, ko_r, _ = ko_model_info

In [10]:
def create_percentage_dict(line_dict):
    highest_coeff = line_dict[0][1]
    percentage_dict = {tf:tfv*100/highest_coeff for tf,tfv in line_dict}
    return percentage_dict

def create_ranked_dict(line_dict):
    ranked_dict = dict(zip([tf for tf,_ in line_dict], range(1, len(line_dict)+1)))
    return ranked_dict

In [11]:
cc_ap, ko_ap, cc_rp, ko_rp = list(map(create_percentage_dict, [cc_a, ko_a, cc_r, ko_r])) 
cc_ar, ko_ar, cc_rr, ko_rr = list(map(create_ranked_dict, [cc_a, ko_a, cc_r, ko_r])) 

# Analysis of model predicted activators and repressors 

We see how activators and repressors change with the KO

In [12]:
all_tf_motifs = list(cc_peaker.featurized_df.columns)

In [13]:
all_tf_motifs = [
    "ZNF143-STAF(Zf)/CUTLL-ZNF143-ChIP-Seq(GSE29600)/Homer|+" if tf=="ZNF143|STAF(Zf)/CUTLL-ZNF143-ChIP-Seq(GSE29600)/Homer|+" else tf for tf in all_tf_motifs]

all_tf_motifs = [
    "ZNF143-STAF(Zf)/CUTLL-ZNF143-ChIP-Seq(GSE29600)/Homer|-" if tf=="ZNF143|STAF(Zf)/CUTLL-ZNF143-ChIP-Seq(GSE29600)/Homer|-" else tf for tf in all_tf_motifs]

In [14]:
motif_meta = "/data5/deepro/starrseq/lib_03162022/results/analysis/activity_prediction/data/homer/metadata_motif.json"

with open(motif_meta, "r") as mm:
    motif_to_genename_dict = json.load(mm)

In [15]:
rnaseq = "/data5/deepro/starrseq/rnaseq/de/diff_exp_to_control/ATF2_vs_CC.tsv"

rnaseq_df = pd.read_csv(rnaseq, sep="\t")

In [16]:
lfc, fdr = rnaseq_df.loc[rnaseq_df.gene_symbol=="TFAP2C", ["logFC", "FDR"]].values.flatten()

In [17]:
tf_info_dict = {}

In [18]:
def get_rnaseq_info(tf_motif):
    motif_name = tf_motif.split("|")[0]
    gene_name = motif_to_genename_dict[motif_name]
    if gene_name:
        tf_rnaseq = rnaseq_df.loc[
            rnaseq_df.gene_symbol==gene_name, ["logFC", "FDR"]
            ]
        if not tf_rnaseq.empty:
            logfc, fdr = tf_rnaseq.values.flatten()
        else:
            logfc, fdr = None, None
    else:
        logfc, fdr = None, None
    return logfc, fdr

In [19]:
for tf in all_tf_motifs:
    if tf in cc_ap:
        if tf in ko_ap:
            change_in_percentage_importance = ko_ap[tf]/cc_ap[tf]
            direction = "aa"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = cc_ar[tf]
            ko_rank = ko_ar[tf]

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr, 
                cc_rank, 
                ko_rank
                )
        
        elif tf in ko_rp:
            change_in_percentage_importance = -ko_rp[tf]/cc_ap[tf]
            direction = "ar"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = cc_ar[tf]
            ko_rank = ko_rr[tf]

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )

        else:
            change_in_percentage_importance = -cc_ap[tf]
            direction = "an"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = cc_ar[tf]
            ko_rank = 1e9

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )

    elif tf in cc_rp:
        if tf in ko_rp:
            change_in_percentage_importance = ko_rp[tf]/cc_rp[tf]
            direction = "rr"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = cc_rr[tf]
            ko_rank = ko_rr[tf]

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )
        
        elif tf in ko_ap:
            change_in_percentage_importance = -ko_ap[tf]/cc_rp[tf]
            direction = "ra"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = cc_rr[tf]
            ko_rank = ko_ar[tf]

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )

        else:
            change_in_percentage_importance = -cc_rp[tf]
            direction = "rn"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = cc_rr[tf]
            ko_rank = 1e9

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )
    else:

        if tf in ko_ap:
            change_in_percentage_importance = ko_ap[tf]
            direction = "na"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = 1e9
            ko_rank = ko_ar[tf]


            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )
        
        elif tf in ko_rp:
            change_in_percentage_importance = ko_rp[tf]
            direction = "nr"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = 1e9
            ko_rank = ko_rr[tf]

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )

        else:
            change_in_percentage_importance = 0
            direction = "nn"
            logfc, fdr = get_rnaseq_info(tf)
            cc_rank = 1e9
            ko_rank = 1e9

            tf_info_dict[tf] = (
                change_in_percentage_importance, 
                direction, 
                logfc, 
                fdr,
                cc_rank, 
                ko_rank
                )                   

In [20]:
len(tf_info_dict)

872

## Filter 1: No Nones

In [21]:
tf_info_dict_no_nones = {tf:val for tf,val in tf_info_dict.items() if val[2]!=None}

In [22]:
len(tf_info_dict_no_nones)

640

## Filter 2: Keep those which have same direction change in both strands


In [23]:
tf_info_dict_consistent_dir = {}

for tf,tfv in tf_info_dict_no_nones.items():

    if tf.split("|")[1] == "+":
        pos_strand_dir = tfv[1]

        tfn = "|".join([tf.split("|")[0], "-"])
        if tfn in tf_info_dict_no_nones:
            neg_strand_dir = tf_info_dict_no_nones[tfn][1]

            if pos_strand_dir == neg_strand_dir:
                tf_info_dict_consistent_dir[tf] = tfv
                tf_info_dict_consistent_dir[tfn] = tf_info_dict_no_nones[tfn] 

In [24]:
len(tf_info_dict_consistent_dir)

382

## Filter 3: Keep those which have the same type of model importance change along both strands

In [25]:
tf_info_dict_consistent_imp_type = {}

for tf,tfv in tf_info_dict_consistent_dir.items():

    if tf.split("|")[1] == "+":
        pos_strand_imp_type = tfv[0]

        tfn = "|".join([tf.split("|")[0], "-"])
        if tfn in tf_info_dict_consistent_dir:
            neg_strand_imp_type = tf_info_dict_consistent_dir[tfn][0]
            
            # if they both have the same sign
            if np.sign(pos_strand_imp_type) == np.sign(neg_strand_imp_type):
                psit, nsit = abs(pos_strand_imp_type), abs(neg_strand_imp_type)

                # if they have both gained or lost importance
                if ((psit<1) and (nsit<1)) or ((psit>1) and (nsit>1)):
                    tf_info_dict_consistent_imp_type[tf] = tfv
                    tf_info_dict_consistent_imp_type[tfn] = tf_info_dict_consistent_dir[tfn] 

In [26]:
len(tf_info_dict_consistent_imp_type)

240

# Cross verifying model with RNASeq

In [28]:
tf_consistent = sorted(list(set([tf.split("|")[0] for tf in tf_info_dict_consistent_imp_type.keys()])))

## DIR - activators that stayed as activators

In [47]:
tf_de_activators = {}

for tf,tfv in tf_info_dict_consistent_imp_type.items():

    if tf.split("|")[1] == "+":
        pos_strand_imp_change = tfv[0]
        pos_strand_dir = tfv[1]

        if pos_strand_dir == "aa":
            if pos_strand_imp_change>1.5:
                tfn = "|".join([tf.split("|")[0], "-"])
                neg_strand_imp_change = tf_info_dict_consistent_imp_type[tfn][0]

                if neg_strand_imp_change>1.5:
                    tf_de_activators[tf] = tfv
                    tf_de_activators[tfn] = tf_info_dict_consistent_imp_type[tfn] 

            elif pos_strand_imp_change<0.5:
                tfn = "|".join([tf.split("|")[0], "-"])
                neg_strand_imp_change = tf_info_dict_consistent_imp_type[tfn][0]

                if neg_strand_imp_change<0.5:
                    tf_de_activators[tf] = tfv
                    tf_de_activators[tfn] = tf_info_dict_consistent_imp_type[tfn] 

In [48]:
tf_de_activators_rnaseq = [tf for tf,tfv in tf_de_activators.items() if tfv[3]<0.05]

In [49]:
len(tf_de_activators), len(tf_de_activators_rnaseq)

(52, 36)

## DIR - repressors that stayed as repressors

In [50]:
tf_de_repressors = {}

for tf,tfv in tf_info_dict_consistent_imp_type.items():

    if tf.split("|")[1] == "+":
        pos_strand_imp_change = tfv[0]
        pos_strand_dir = tfv[1]

        if pos_strand_dir == "rr":
            if pos_strand_imp_change>1.5:
                tfn = "|".join([tf.split("|")[0], "-"])
                neg_strand_imp_change = tf_info_dict_consistent_imp_type[tfn][0]

                if neg_strand_imp_change>1.5:
                    tf_de_repressors[tf] = tfv
                    tf_de_repressors[tfn] = tf_info_dict_consistent_imp_type[tfn] 

            elif pos_strand_imp_change<0.5:
                tfn = "|".join([tf.split("|")[0], "-"])
                neg_strand_imp_change = tf_info_dict_consistent_imp_type[tfn][0]

                if neg_strand_imp_change<0.5:
                    tf_de_repressors[tf] = tfv
                    tf_de_repressors[tfn] = tf_info_dict_consistent_imp_type[tfn] 

In [52]:
tf_de_repressors_rnaseq = [tf for tf,tfv in tf_de_repressors.items() if tfv[3]<0.05]

In [53]:
len(tf_de_repressors), len(tf_de_repressors_rnaseq)

(18, 18)